# 06-A. legacy LightGBM failure audit

이 노트북은 기존 LightGBM 위험도 모델 이후에 `split_event_based`와 normal 기준이 어떤 분포 차이를 만드는지 점검한다.

목적:
- holdout normal이 왜 높은 risk를 받는지 확인
- 제조사, 기계실, fault label, lead time, feature drift 기준으로 split 차이 진단
- `manufacturer 2 / SH` holdout 붕괴가 모델 튜닝 문제인지 라벨/시간/normal 기준 문제인지 분리
- 왜 이 legacy 방식이 canonical이 되기 어려운지 failure evidence로 보존


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for path in [current, *current.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('pyproject.toml을 찾을 수 없습니다. 프로젝트 루트에서 실행해 주세요.')


PROJECT_ROOT = find_project_root(Path.cwd())
RISK_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml_risk'
FEATURE_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml_features'
WINDOW_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml_windows'
ALIGNMENT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'label_alignment'

RISK_SCORES_PATH = RISK_DIR / 'lgbm_risk_scores.csv'
FEATURE_TABLE_PATH = FEATURE_DIR / 'trainable_windows.csv'
WINDOW_DATASET_PATH = WINDOW_DIR / 'ml_window_dataset.csv'
FAULT_ALIGNMENT_PATH = ALIGNMENT_DIR / 'fault_alignment.csv'
DISTURBANCE_ALIGNMENT_PATH = ALIGNMENT_DIR / 'disturbance_alignment.csv'
MODEL_METADATA_PATH = RISK_DIR / 'models' / 'risk_model_metadata.json'

SPLIT_LABEL_DIAGNOSTICS_PATH = RISK_DIR / 'holdout_split_label_diagnostics.csv'
ERROR_DIAGNOSTICS_PATH = RISK_DIR / 'holdout_error_diagnostics.csv'
FEATURE_DRIFT_DIAGNOSTICS_PATH = RISK_DIR / 'holdout_feature_drift_diagnostics.csv'
GROUP_CALIBRATION_DIAGNOSTICS_PATH = RISK_DIR / 'holdout_group_calibration_diagnostics.csv'
GROUP_THRESHOLD_DIAGNOSTICS_PATH = RISK_DIR / 'holdout_group_threshold_diagnostics.csv'
LABEL_TIME_DIAGNOSTICS_PATH = RISK_DIR / 'holdout_label_time_diagnostics.csv'
M2_SH_LABEL_TIME_DIAGNOSTICS_PATH = RISK_DIR / 'manufacturer2_sh_label_time_diagnostics.csv'
M2_SH_SUBSTATION_SUMMARY_PATH = RISK_DIR / 'manufacturer2_sh_substation_label_summary.csv'

print(RISK_SCORES_PATH)
print(FEATURE_TABLE_PATH)


C:\Project3\HeatGrid_Agent\data\processed\ml_risk\lgbm_risk_scores.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_features\trainable_windows.csv


## 1. 입력 로딩

06번 risk score table과 04번 feature table, 모델 metadata를 함께 읽는다.


In [2]:
risk_scores = pd.read_csv(RISK_SCORES_PATH)
feature_table = pd.read_csv(FEATURE_TABLE_PATH)
model_metadata = json.loads(MODEL_METADATA_PATH.read_text(encoding='utf-8'))

risk_scores['pred_high'] = risk_scores['risk_level'].isin(['high', 'critical']).astype(int)
risk_scores['target'] = (risk_scores['label'] == 'pre_fault').astype(int)

display(risk_scores.head())
display(pd.crosstab(risk_scores['split_event_based'], risk_scores['label']))
display(pd.crosstab([risk_scores['split_event_based'], risk_scores['manufacturer']], risk_scores['label']))


,manufacturer,substation_id,source_file,window_start,window_end,label,fault_label,fault_event_id,configuration_type,estimated_lead_time_hours,lead_time_bucket,anomaly_score,anomaly_threshold,anomaly_label,risk_score,risk_probability,risk_level,main_abnormal_features,related_fault_history,related_disturbance_history,model_explanation_features,maintenance_related,disturbance_count,days_since_last_fault_event,has_previous_fault_event,recent_fault_30d,recent_fault_60d,recent_fault_90d,days_since_last_task_event,has_previous_task_event,recent_task_30d,recent_task_60d,recent_task_90d,days_since_last_any_event,has_previous_any_event,recent_any_30d,recent_any_60d,recent_any_90d,post_fault_recent_90d,post_task_recent_90d,split_event_based,split_event_regime_based,split_regime_based,split_time_based,split_substation_based,model_version,pred_high,target
0,manufacturer 1,1,substation_1.csv,2019-12-01 00:00:00,2019-12-01 06:00:00,normal,NaN,NaN,SH + DHW,NaN,NaN,0.416895,0.535778,0,0.156220,0.156220,low,s_dhw_lower_storage_temperature__max|dhw_suppl...,[],[],days_since_last_any_event|days_since_last_task...,False,0,9999.0,0,0,0,0,9999.0,0,0,0,0,9999.0,0,0,0,0,0,0,validation,train,train,validation,holdout,lgbm_risk_06_event_days_v3,0,0
1,manufacturer 1,1,substation_1.csv,2019-12-01 06:00:00,2019-12-01 12:00:00,normal,NaN,NaN,SH + DHW,NaN,NaN,0.430012,0.535778,0,0.090324,0.090324,low,s_dhw_supply_temperature_setpoint__delta|dhw_s...,[],[],days_since_last_any_event|days_since_last_task...,False,0,9999.0,0,0,0,0,9999.0,0,0,0,0,9999.0,0,0,0,0,0,0,validation,train,train,validation,holdout,lgbm_risk_06_event_days_v3,0,0
2,manufacturer 1,1,substation_1.csv,2019-12-01 18:00:00,2019-12-02 00:00:00,normal,NaN,NaN,SH + DHW,NaN,NaN,0.426032,0.535778,0,0.108072,0.108072,low,p_net_return_temperature__std|s_hc1_supply_tem...,[],[],days_since_last_any_event|days_since_last_task...,False,0,9999.0,0,0,0,0,9999.0,0,0,0,0,9999.0,0,0,0,0,0,0,validation,train,train,validation,holdout,lgbm_risk_06_event_days_v3,0,0
3,manufacturer 1,1,substation_1.csv,2019-12-02 12:00:00,2019-12-02 18:00:00,normal,NaN,NaN,SH + DHW,NaN,NaN,0.440612,0.535778,0,0.413477,0.413477,high,s_dhw_supply_temperature_setpoint__delta|p_net...,[],[],days_since_last_any_event|days_since_last_task...,False,0,9999.0,0,0,0,0,9999.0,0,0,0,0,9999.0,0,0,0,0,0,0,validation,validation,validation,validation,holdout,lgbm_risk_06_event_days_v3,1,0
4,manufacturer 1,1,substation_1.csv,2019-12-02 18:00:00,2019-12-03 00:00:00,normal,NaN,NaN,SH + DHW,NaN,NaN,0.424733,0.535778,0,0.119385,0.119385,low,days_since_last_task_event|days_since_last_any...,[],[],days_since_last_any_event|days_since_last_task...,False,0,9999.0,0,0,0,0,9999.0,0,0,0,0,9999.0,0,0,0,0,0,0,validation,validation,validation,validation,holdout,lgbm_risk_06_event_days_v3,0,0


label,normal,pre_fault
split_event_based,,
holdout,251,86
train,1235,407
validation,240,143


label                             normal  pre_fault
split_event_based manufacturer                     
holdout           manufacturer 1     106         41
                  manufacturer 2     145         45
train             manufacturer 1     682        188
                  manufacturer 2     553        219
validation        manufacturer 1     143         66
                  manufacturer 2      97         77

## 2. split / label / manufacturer 진단

holdout normal과 pre_fault가 제조사별로 어떻게 다르게 보이는지 저장한다.


In [3]:
split_label_diagnostics = (
    risk_scores.groupby(['split_event_based', 'manufacturer', 'label'])
    .agg(
        rows=('risk_probability', 'size'),
        mean_risk=('risk_probability', 'mean'),
        median_risk=('risk_probability', 'median'),
        high_or_critical_rate=('pred_high', 'mean'),
        substation_count=('substation_id', 'nunique'),
    )
    .reset_index()
)
split_label_diagnostics.to_csv(SPLIT_LABEL_DIAGNOSTICS_PATH, index=False)

display(split_label_diagnostics)


,split_event_based,manufacturer,label,rows,mean_risk,median_risk,high_or_critical_rate,substation_count
0,holdout,manufacturer 1,normal,106,0.320524,0.222775,0.311321,4
1,holdout,manufacturer 1,pre_fault,41,0.673084,0.665056,0.780488,5
2,holdout,manufacturer 2,normal,145,0.195756,0.123448,0.158621,7
3,holdout,manufacturer 2,pre_fault,45,0.373060,0.254614,0.377778,4
4,train,manufacturer 1,normal,682,0.024450,0.014505,0.008798,16
5,train,manufacturer 1,pre_fault,188,0.975217,0.980482,1.000000,13
6,train,manufacturer 2,normal,553,0.048683,0.016911,0.019892,13
7,train,manufacturer 2,pre_fault,219,0.977139,0.980547,1.000000,15
8,validation,manufacturer 1,normal,143,0.155253,0.108072,0.041958,7
9,validation,manufacturer 1,pre_fault,66,0.733005,0.959016,0.803030,5


## 3. false positive / false negative 진단

어떤 substation과 fault event에 오류가 몰리는지 확인한다.


In [4]:
error_rows = []
for split_name, split_df in risk_scores.groupby('split_event_based'):
    false_positive = split_df[(split_df['target'] == 0) & (split_df['pred_high'] == 1)]
    false_negative = split_df[(split_df['target'] == 1) & (split_df['pred_high'] == 0)]
    error_rows.append(
        {
            'split_event_based': split_name,
            'rows': len(split_df),
            'false_positive_count': len(false_positive),
            'false_negative_count': len(false_negative),
            'false_positive_substations': '|'.join(map(str, false_positive['substation_id'].value_counts().head(10).index.tolist())),
            'false_negative_fault_events': '|'.join(map(str, false_negative['fault_event_id'].value_counts().head(10).index.tolist())),
        }
    )

error_diagnostics = pd.DataFrame(error_rows)
error_diagnostics.to_csv(ERROR_DIAGNOSTICS_PATH, index=False)

display(error_diagnostics)


,split_event_based,rows,false_positive_count,false_negative_count,false_positive_substations,false_negative_fault_events
0,holdout,337,56,37,6|25|11|59|53|43,2.0|3.0|10.0|1.0|53.0|69.0
1,train,1642,17,0,40|2,
2,validation,383,17,37,53|13|22|1,6.0|5.0|15.0|23.0


## 4-A. group calibration 진단

threshold 조정만으로 holdout false positive가 줄어드는지 확인한다.
같은 holdout 집합에서 global threshold, group F1 threshold, validation normal 분위수 threshold를 비교한다.


In [5]:
threshold_grid = np.round(np.arange(0.05, 0.951, 0.025), 3)
base_threshold = float(model_metadata['risk_thresholds']['high'])
risk_scores['group_key'] = risk_scores['manufacturer'].astype(str) + '|' + risk_scores['configuration_type'].astype(str)

def binary_metrics(frame: pd.DataFrame, prediction_column: str) -> dict:
    y_true = (frame['label'] == 'pre_fault').astype(int).to_numpy()
    y_pred = frame[prediction_column].astype(int).to_numpy()
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', zero_division=0
    )
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    return {
        'rows': len(frame),
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'false_positive_rate': fp / (fp + tn) if fp + tn else 0.0,
    }

validation_scores = risk_scores[risk_scores['split_event_based'] == 'validation'].copy()
holdout_scores = risk_scores[risk_scores['split_event_based'] == 'holdout'].copy()

threshold_rows = []
threshold_lookup = {'global': {}}
threshold_lookup['group_f1'] = {}
threshold_lookup['group_normal_p90'] = {}
threshold_lookup['group_normal_p95'] = {}

for group_key, group in validation_scores.groupby('group_key'):
    y_true = (group['label'] == 'pre_fault').astype(int)
    selected_f1_threshold = base_threshold
    selected_f1 = np.nan
    if y_true.nunique() == 2:
        best_threshold, best_f1 = base_threshold, -1.0
        for threshold in threshold_grid:
            y_pred = (group['risk_probability'] >= threshold).astype(int)
            f1 = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)[2]
            if f1 > best_f1:
                best_threshold, best_f1 = float(threshold), float(f1)
        selected_f1_threshold = best_threshold
        selected_f1 = best_f1
    normal_scores = group.loc[group['label'] == 'normal', 'risk_probability']
    normal_p90 = max(base_threshold, float(normal_scores.quantile(0.90))) if len(normal_scores) else base_threshold
    normal_p95 = max(base_threshold, float(normal_scores.quantile(0.95))) if len(normal_scores) else base_threshold
    threshold_lookup['group_f1'][group_key] = selected_f1_threshold
    threshold_lookup['group_normal_p90'][group_key] = normal_p90
    threshold_lookup['group_normal_p95'][group_key] = normal_p95
    threshold_rows.append({
        'group_key': group_key,
        'validation_rows': len(group),
        'validation_normal_count': int((group['label'] == 'normal').sum()),
        'validation_prefault_count': int((group['label'] == 'pre_fault').sum()),
        'global_threshold': base_threshold,
        'group_f1_threshold': selected_f1_threshold,
        'group_f1_validation_f1': selected_f1,
        'group_normal_p90_threshold': normal_p90,
        'group_normal_p95_threshold': normal_p95,
    })

calibrated_holdout = holdout_scores.copy()
calibrated_holdout['pred_global'] = calibrated_holdout['risk_probability'] >= base_threshold
for strategy in ['group_f1', 'group_normal_p90', 'group_normal_p95']:
    calibrated_holdout[f'pred_{strategy}'] = calibrated_holdout.apply(
        lambda row: row['risk_probability'] >= threshold_lookup[strategy].get(row['group_key'], base_threshold),
        axis=1,
    )

calibration_rows = []
for prediction_column in ['pred_global', 'pred_group_f1', 'pred_group_normal_p90', 'pred_group_normal_p95']:
    row = binary_metrics(calibrated_holdout, prediction_column)
    row['strategy'] = prediction_column.replace('pred_', '')
    calibration_rows.append(row)

group_threshold_diagnostics = pd.DataFrame(threshold_rows).sort_values('group_key')
group_calibration_diagnostics = pd.DataFrame(calibration_rows)[
    ['strategy', 'rows', 'tp', 'fp', 'fn', 'tn', 'precision', 'recall', 'f1', 'false_positive_rate']
]

group_threshold_diagnostics.to_csv(GROUP_THRESHOLD_DIAGNOSTICS_PATH, index=False)
group_calibration_diagnostics.to_csv(GROUP_CALIBRATION_DIAGNOSTICS_PATH, index=False)

display(group_threshold_diagnostics)
display(group_calibration_diagnostics)


,group_key,validation_rows,validation_normal_count,validation_prefault_count,global_threshold,group_f1_threshold,group_f1_validation_f1,group_normal_p90_threshold,group_normal_p95_threshold
0,manufacturer 1|SH + DHW,175,112,63,0.4,0.425,0.879310,0.400000,0.400000
1,manufacturer 1|SH + DHW with sub-circuits,34,31,3,0.4,0.325,0.800000,0.400000,0.400000
2,manufacturer 2|SH,104,53,51,0.4,0.150,0.866667,0.400000,0.400000
3,manufacturer 2|SH + DHW,14,14,0,0.4,0.400,NaN,0.400000,0.400000
4,manufacturer 2|SH with buffer tank,56,30,26,0.4,0.425,0.560000,0.691023,0.698472


,strategy,rows,tp,fp,fn,tn,precision,recall,f1,false_positive_rate
0,global,337,49,56,37,195,0.466667,0.569767,0.513089,0.223108
1,group_f1,337,52,68,34,183,0.433333,0.604651,0.504854,0.270916
2,group_normal_p90,337,46,54,40,197,0.460000,0.534884,0.494624,0.215139
3,group_normal_p95,337,46,54,40,197,0.460000,0.534884,0.494624,0.215139


## 4-B. manufacturer 2 / SH label-time 진단

`manufacturer 2 / SH`에서 holdout normal risk가 pre_fault보다 높게 나오는 원인을 라벨, 기계실, 시간, 최근 이벤트 관점에서 고정 산출물로 확인한다.


In [6]:
for column in ['window_start', 'window_end']:
    risk_scores[column] = pd.to_datetime(risk_scores[column], errors='coerce')
    feature_table[column] = pd.to_datetime(feature_table[column], errors='coerce')

fault_alignment = pd.read_csv(FAULT_ALIGNMENT_PATH)
disturbance_alignment = pd.read_csv(DISTURBANCE_ALIGNMENT_PATH)
for column in ['report_date', 'window_start', 'window_end']:
    if column in fault_alignment.columns:
        fault_alignment[column] = pd.to_datetime(fault_alignment[column], errors='coerce')
for column in ['event_start', 'window_start', 'window_end']:
    if column in disturbance_alignment.columns:
        disturbance_alignment[column] = pd.to_datetime(disturbance_alignment[column], errors='coerce')

key_columns = ['manufacturer', 'substation_id', 'window_start', 'window_end']
label_time_columns = [
    'normal_event_related',
    'maintenance_related',
    'data_quality_issue',
    'normal_reference_outlier',
    'normal_reference_outlier_count',
    'normal_reference_filter_reason',
    'window_source_type',
    'p_net_return_temperature__mean',
    'p_net_return_temperature__std',
    's_hc1_supply_temperature_setpoint__std',
    'network_temperature_gap__mean',
    'outdoor_temperature__mean',
]
label_time_columns = [column for column in label_time_columns if column in feature_table.columns]
label_time_df = risk_scores.merge(
    feature_table[key_columns + label_time_columns],
    on=key_columns,
    how='left',
    suffixes=('', '_feature'),
)


def nearest_fault(row: pd.Series) -> pd.Series:
    events = fault_alignment[
        fault_alignment['manufacturer'].eq(row['manufacturer'])
        & fault_alignment['substation_id'].eq(row['substation_id'])
        & fault_alignment['report_date'].notna()
    ].copy()
    if events.empty:
        return pd.Series(
            {
                'nearest_fault_event_id': pd.NA,
                'nearest_fault_report_date': pd.NaT,
                'nearest_fault_days_abs': pd.NA,
                'nearest_fault_signed_days': pd.NA,
                'nearest_fault_label': pd.NA,
                'nearest_fault_problem': pd.NA,
            }
        )
    events['nearest_fault_signed_days'] = (events['report_date'] - row['window_end']).dt.total_seconds() / 86400
    events['nearest_fault_days_abs'] = events['nearest_fault_signed_days'].abs()
    nearest = events.sort_values(['nearest_fault_days_abs', 'report_date']).iloc[0]
    return pd.Series(
        {
            'nearest_fault_event_id': nearest.get('event_id', pd.NA),
            'nearest_fault_report_date': nearest.get('report_date', pd.NaT),
            'nearest_fault_days_abs': nearest.get('nearest_fault_days_abs', pd.NA),
            'nearest_fault_signed_days': nearest.get('nearest_fault_signed_days', pd.NA),
            'nearest_fault_label': nearest.get('fault_label', pd.NA),
            'nearest_fault_problem': nearest.get('problem', pd.NA),
        }
    )


def nearest_disturbance(row: pd.Series) -> pd.Series:
    events = disturbance_alignment[
        disturbance_alignment['manufacturer'].eq(row['manufacturer'])
        & disturbance_alignment['substation_id'].eq(row['substation_id'])
        & disturbance_alignment['event_start'].notna()
    ].copy()
    if events.empty:
        return pd.Series(
            {
                'nearest_disturbance_type': pd.NA,
                'nearest_disturbance_start': pd.NaT,
                'nearest_disturbance_days_abs': pd.NA,
                'nearest_disturbance_signed_days': pd.NA,
            }
        )
    events['nearest_disturbance_signed_days'] = (events['event_start'] - row['window_end']).dt.total_seconds() / 86400
    events['nearest_disturbance_days_abs'] = events['nearest_disturbance_signed_days'].abs()
    nearest = events.sort_values(['nearest_disturbance_days_abs', 'event_start']).iloc[0]
    return pd.Series(
        {
            'nearest_disturbance_type': nearest.get('disturbance_type', pd.NA),
            'nearest_disturbance_start': nearest.get('event_start', pd.NaT),
            'nearest_disturbance_days_abs': nearest.get('nearest_disturbance_days_abs', pd.NA),
            'nearest_disturbance_signed_days': nearest.get('nearest_disturbance_signed_days', pd.NA),
        }
    )

m2_sh = label_time_df[
    label_time_df['manufacturer'].eq('manufacturer 2')
    & label_time_df['configuration_type'].eq('SH')
].copy()
nearest_fault_df = m2_sh.apply(nearest_fault, axis=1)
nearest_disturbance_df = m2_sh.apply(nearest_disturbance, axis=1)
m2_sh_diagnostics = pd.concat(
    [m2_sh.reset_index(drop=True), nearest_fault_df.reset_index(drop=True), nearest_disturbance_df.reset_index(drop=True)],
    axis=1,
)

m2_sh_export_columns = [
    'manufacturer',
    'configuration_type',
    'substation_id',
    'split_event_based',
    'split_time_based',
    'label',
    'window_start',
    'window_end',
    'window_source_type',
    'fault_event_id',
    'fault_label',
    'estimated_lead_time_hours',
    'lead_time_bucket',
    'risk_probability',
    'risk_level',
    'anomaly_score',
    'normal_event_related',
    'maintenance_related',
    'disturbance_count',
    'data_quality_issue',
    'normal_reference_outlier',
    'normal_reference_outlier_count',
    'normal_reference_filter_reason',
    'p_net_return_temperature__mean',
    'p_net_return_temperature__std',
    's_hc1_supply_temperature_setpoint__std',
    'network_temperature_gap__mean',
    'outdoor_temperature__mean',
    'nearest_fault_event_id',
    'nearest_fault_report_date',
    'nearest_fault_days_abs',
    'nearest_fault_signed_days',
    'nearest_fault_label',
    'nearest_fault_problem',
    'nearest_disturbance_type',
    'nearest_disturbance_start',
    'nearest_disturbance_days_abs',
    'nearest_disturbance_signed_days',
]
m2_sh_export_columns = [column for column in m2_sh_export_columns if column in m2_sh_diagnostics.columns]
m2_sh_diagnostics[m2_sh_export_columns].sort_values(
    ['split_event_based', 'label', 'substation_id', 'window_start']
).to_csv(M2_SH_LABEL_TIME_DIAGNOSTICS_PATH, index=False, encoding='utf-8-sig')

m2_sh_summary = (
    m2_sh_diagnostics.groupby(['split_event_based', 'label', 'substation_id'], dropna=False)
    .agg(
        rows=('risk_probability', 'size'),
        risk_mean=('risk_probability', 'mean'),
        risk_median=('risk_probability', 'median'),
        risk_min=('risk_probability', 'min'),
        risk_max=('risk_probability', 'max'),
        high_or_critical_rate=('pred_high', 'mean'),
        anomaly_mean=('anomaly_score', 'mean'),
        normal_reference_outlier_rate=('normal_reference_outlier', 'mean'),
        normal_reference_outlier_count_mean=('normal_reference_outlier_count', 'mean'),
        p_net_return_temperature_mean=('p_net_return_temperature__mean', 'mean'),
        p_net_return_temperature_std_mean=('p_net_return_temperature__std', 'mean'),
        setpoint_std_mean=('s_hc1_supply_temperature_setpoint__std', 'mean'),
        outdoor_temperature_mean=('outdoor_temperature__mean', 'mean'),
        nearest_fault_days_abs_min=('nearest_fault_days_abs', 'min'),
        nearest_fault_signed_days_median=('nearest_fault_signed_days', 'median'),
        window_start_min=('window_start', 'min'),
        window_start_max=('window_start', 'max'),
    )
    .reset_index()
)
m2_sh_summary.to_csv(M2_SH_SUBSTATION_SUMMARY_PATH, index=False, encoding='utf-8-sig')

holdout_label_time = label_time_df[label_time_df['split_event_based'].eq('holdout')].copy()
holdout_nearest_fault = holdout_label_time.apply(nearest_fault, axis=1)
holdout_label_time = pd.concat([holdout_label_time.reset_index(drop=True), holdout_nearest_fault.reset_index(drop=True)], axis=1)
holdout_label_time_summary = (
    holdout_label_time.groupby(['manufacturer', 'configuration_type', 'label'], dropna=False)
    .agg(
        rows=('risk_probability', 'size'),
        substation_count=('substation_id', 'nunique'),
        risk_mean=('risk_probability', 'mean'),
        risk_median=('risk_probability', 'median'),
        high_or_critical_rate=('pred_high', 'mean'),
        normal_reference_outlier_rate=('normal_reference_outlier', 'mean'),
        nearest_fault_days_abs_median=('nearest_fault_days_abs', 'median'),
        window_start_min=('window_start', 'min'),
        window_start_max=('window_start', 'max'),
    )
    .reset_index()
    .sort_values(['risk_mean', 'rows'], ascending=[False, False])
)
holdout_label_time_summary.to_csv(LABEL_TIME_DIAGNOSTICS_PATH, index=False, encoding='utf-8-sig')

display(m2_sh_summary)
display(holdout_label_time_summary.head(20))


,split_event_based,label,substation_id,rows,risk_mean,risk_median,risk_min,risk_max,high_or_critical_rate,anomaly_mean,normal_reference_outlier_rate,normal_reference_outlier_count_mean,p_net_return_temperature_mean,p_net_return_temperature_std_mean,setpoint_std_mean,outdoor_temperature_mean,nearest_fault_days_abs_min,nearest_fault_signed_days_median,window_start_min,window_start_max
0,holdout,normal,11,26,0.487695,0.635305,0.040180,0.871992,0.615385,0.422895,0.0,0.576923,39.099359,3.454985,3.131831,8.266453,90.709028,-94.084028,2020-03-06 00:00:00,2020-03-12 18:00:00
1,holdout,normal,59,20,0.246485,0.205532,0.041469,0.519516,0.200000,0.417929,0.0,0.550000,40.615278,4.562576,2.426426,9.072917,42.634722,-45.759722,2020-03-07 00:00:00,2020-03-13 18:00:00
2,holdout,pre_fault,45,12,0.667711,0.753218,0.227134,0.964937,0.750000,0.470037,0.0,0.000000,48.589597,3.616317,4.482788,5.582761,0.008333,1.383333,2020-03-06 12:00:00,2020-03-09 06:00:00
3,train,normal,4,56,0.012881,0.008175,0.005516,0.178016,0.000000,0.425542,0.0,0.035714,59.252480,1.924520,0.571521,14.089236,462.838194,-547.213194,2018-07-15 00:00:00,2018-12-30 18:00:00
4,train,normal,9,28,0.014667,0.010286,0.003227,0.083554,0.000000,0.435235,0.0,0.071429,45.960317,2.966811,0.502018,4.645139,NaN,NaN,2018-12-14 00:00:00,2018-12-20 18:00:00
5,train,normal,10,56,0.076460,0.039846,0.018622,0.303229,0.000000,0.518924,0.0,0.000000,63.373512,1.457599,0.960868,23.233780,58.430556,-154.305556,2018-02-15 00:00:00,2018-08-25 18:00:00
6,train,normal,12,28,0.019968,0.014166,0.009709,0.055028,0.000000,0.431195,0.0,0.000000,53.303571,1.708995,0.616462,7.387897,NaN,NaN,2018-01-14 00:00:00,2018-01-20 18:00:00
7,train,pre_fault,4,34,0.979481,0.981296,0.933677,0.993486,1.000000,0.432962,0.0,0.000000,51.989706,2.227846,1.160993,7.866748,0.110417,1.411458,2016-07-29 12:00:00,2017-03-12 00:00:00
8,train,pre_fault,10,50,0.986447,0.991110,0.850614,0.992576,1.000000,0.525676,0.0,0.000000,59.924230,2.064648,0.552271,22.873687,0.069444,2.078819,2016-09-26 12:00:00,2019-10-14 00:00:00
9,train,pre_fault,11,2,0.935893,0.935893,0.934018,0.937769,1.000000,0.496303,0.0,0.000000,21.263889,1.532727,5.066415,6.540278,0.164583,0.289583,2019-11-25 18:00:00,2019-11-26 00:00:00


,manufacturer,configuration_type,label,rows,substation_count,risk_mean,risk_median,high_or_critical_rate,normal_reference_outlier_rate,nearest_fault_days_abs_median,window_start_min,window_start_max
1,manufacturer 1,SH + DHW,pre_fault,27,4,0.731658,0.918753,0.740741,0.0,1.166667,2014-05-03 18:00:00,2020-05-26 00:00:00
5,manufacturer 2,SH,pre_fault,12,1,0.667711,0.753218,0.750000,0.0,1.383333,2020-03-06 12:00:00,2020-03-09 06:00:00
3,manufacturer 1,SH + DHW with sub-circuits,pre_fault,14,1,0.560121,0.596479,0.857143,0.0,1.034028,2020-03-07 18:00:00,2020-06-13 00:00:00
0,manufacturer 1,SH + DHW,normal,78,3,0.414303,0.353528,0.423077,0.0,416.346528,2020-03-14 00:00:00,2020-06-08 18:00:00
4,manufacturer 2,SH,normal,46,2,0.382821,0.334535,0.434783,0.0,91.334028,2020-03-06 00:00:00,2020-03-13 18:00:00
8,manufacturer 2,SH with buffer tank,normal,5,1,0.357194,0.328068,0.400000,0.0,20.916667,2020-02-23 18:00:00,2020-02-24 18:00:00
7,manufacturer 2,SH + DHW,pre_fault,11,1,0.349420,0.347696,0.454545,0.0,1.465972,2020-03-15 12:00:00,2020-03-18 00:00:00
9,manufacturer 2,SH with buffer tank,pre_fault,22,2,0.224162,0.175284,0.136364,0.0,1.435069,2020-01-18 12:00:00,2020-03-16 00:00:00
6,manufacturer 2,SH + DHW,normal,94,4,0.095625,0.028146,0.010638,0.0,110.0,2020-02-23 12:00:00,2020-03-20 18:00:00
2,manufacturer 1,SH + DHW with sub-circuits,normal,28,1,0.059282,0.047416,0.000000,0.0,1032.309722,2020-04-01 00:00:00,2020-04-07 18:00:00


## 4. holdout feature drift 진단

train normal 대비 holdout normal과 holdout pre_fault의 평균 차이를 z-score 기준으로 본다.


In [7]:
model_feature_columns = model_metadata['model_feature_columns']
key_columns = ['manufacturer', 'substation_id', 'window_start', 'window_end']
feature_columns = [column for column in model_feature_columns if column in feature_table.columns]
risk_columns = [column for column in model_feature_columns if column in risk_scores.columns]

diagnostic_df = risk_scores[key_columns + ['split_event_based', 'label', 'risk_probability'] + risk_columns].merge(
    feature_table[key_columns + feature_columns],
    on=key_columns,
    how='left',
)

train_normal = diagnostic_df[(diagnostic_df['split_event_based'] == 'train') & (diagnostic_df['label'] == 'normal')]
holdout_normal = diagnostic_df[(diagnostic_df['split_event_based'] == 'holdout') & (diagnostic_df['label'] == 'normal')]
holdout_prefault = diagnostic_df[(diagnostic_df['split_event_based'] == 'holdout') & (diagnostic_df['label'] == 'pre_fault')]

drift_rows = []
for column in model_feature_columns:
    if column not in diagnostic_df.columns:
        continue
    train_mean = train_normal[column].mean()
    train_std = train_normal[column].std()
    if pd.isna(train_std) or train_std == 0:
        train_std = 1.0
    holdout_normal_mean = holdout_normal[column].mean()
    holdout_prefault_mean = holdout_prefault[column].mean()
    drift_rows.append(
        {
            'feature': column,
            'train_normal_mean': train_mean,
            'holdout_normal_mean': holdout_normal_mean,
            'holdout_prefault_mean': holdout_prefault_mean,
            'holdout_normal_z_vs_train': abs((holdout_normal_mean - train_mean) / train_std),
            'holdout_prefault_z_vs_train': abs((holdout_prefault_mean - train_mean) / train_std),
        }
    )

feature_drift_diagnostics = pd.DataFrame(drift_rows).sort_values('holdout_normal_z_vs_train', ascending=False)
feature_drift_diagnostics.to_csv(FEATURE_DRIFT_DIAGNOSTICS_PATH, index=False)

display(feature_drift_diagnostics.head(30))


,feature,train_normal_mean,holdout_normal_mean,holdout_prefault_mean,holdout_normal_z_vs_train,holdout_prefault_z_vs_train
47,season_bucket__is__spring,0.140081,0.824701,0.779070,1.971766,1.840344
3,doy_sin,-0.073417,0.893586,0.734892,1.380464,1.153917
184,days_since_last_fault_event,7804.566457,2902.384161,7890.271302,1.208752,0.021133
178,day_of_year,182.231579,84.960159,89.383721,0.808383,0.771620
103,p_net_return_temperature__mean,51.662591,45.086633,43.547287,0.802956,0.990918
101,p_net_return_temperature__last,51.866273,44.882470,43.230136,0.797435,0.986105
182,month,6.519838,3.446215,3.476744,0.785911,0.778104
100,p_net_return_temperature__first,51.762750,45.064502,43.755814,0.755140,0.902677
104,p_net_return_temperature__min,45.094360,38.187649,36.813469,0.670156,0.803492
49,season_bucket__is__winter,0.403239,0.075697,0.127907,0.667435,0.561047


## 5. 핵심 해석

이 노트북의 목적은 모델을 다시 학습하는 것이 아니라, holdout 붕괴 원인이 split/normal 기준 문제인지 feature drift 문제인지 분리하는 것이다.

현재 결과는 다음 해석을 지원한다.
- holdout normal이 train normal과 다른 분포를 가진다.
- 특히 `manufacturer 2 / SH` holdout normal은 `substation 11, 59`의 2020년 3월 normal context이며, holdout pre_fault는 `substation 45`의 단일 fault event다.
- `manufacturer 2 / SH` holdout normal은 과거 fault/task 이후 구간이고, train normal 대비 `p_net_return_temperature`와 setpoint 변동성이 크게 다르다.
- threshold 조정이나 group별 calibration만으로는 false positive를 안정적으로 줄이지 못한다.
- 다음 단계는 07 explainability가 아니라 06 안에서 label/normal/time 정의를 보강하는 것이다.


## 2026-06-25 promotion note

- overall winner: `thermal_group_zscore_only`
- manufacturer 2 / SH winner: `event_context_only`
- hybrid promoted candidate was tested, but overall holdout was worse than current calibrated official chain
- keep official downstream input as `data/processed/ml_risk/lgbm_risk_scores_calibrated.csv` with `risk_level_calibrated`
- promotion review doc: `PREPROCESSING/docs/06_promotion_decision.md`
